In [1]:
# This script performs statistical analysis of the TRF results

from pathlib import Path
import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.formula.api as smf

# Define the path that will be used
DATA_ROOT = Path('/Users/zorkabozilovic/Desktop/PART1')
df = pd.read_csv(DATA_ROOT / 'all_trf_results.csv')

# Constants
bands = ['delta', 'theta', 'alpha', 'beta', 'wideband']
feat_orders = {
    'not standardized': ['envelope', 'onsets', 'pitch', 'centroid', 'mel'],
    'standardized': ['envelope', 'onsets', 'pitch', 'centroid', 'mfcc3'],
}

In [2]:
# Exclude rows where boosting learned no TRF kernel (check checking_nans.ipynb for more details)
# Excluded are: Sub2 centroid decoding for alpha and beta and both pipelines, Sub19 pitch decoding for alpha and raw (not standardized) pipeline only

before = len(df)
df = df.dropna(subset=['mean_r'])
print(f'Excluded {before - len(df)} rows with NaN values')

Excluded 5 rows with NaN values


In [3]:
# Two-way ANOVA with group (musicians vs non-musicians) and band (delta, theta, alpha, beta, wideband) as factors (for each feature decoding for both pipelines)

print('Two-way ANOVA: group * band for each feature decoding and both pipelines')

# Function for building long format from all trials in the CSV file
def to_long(data):
    rows = []
    for _, row in data.iterrows():
        
        # For all trials
        for t in range(1, 31):
            val = row[f'r_trial_{t}']
            
            # Add all needed info for ANOVA (trail r's, subject,group and band)
            rows.append({
                'subject': row['subject'],
                'group': row['group'], 
                'band': row['band'],
                'r': val,
            })
    return pd.DataFrame(rows)

# Both pipelines
for pipeline in ['not standardized', 'standardized']:
    
    # Use the same feature order as before
    feats = feat_orders[pipeline]
    
    # Get decoding rows
    dec_df = df[(df['direction'] == 'decoding') & (df['pipeline'] == pipeline)]
    
    # One ANOVA per single feature
    for feat in feats:
        feat_df = dec_df[dec_df['combo_name'] == feat]
        
        # Reshape to long format (one row per trial)
        long = to_long(feat_df)
        
        # Print the result for this (feature and pipeline)
        print(f'\n  {feat} — {pipeline}')
        try:
            # Two-way ANOVA: r ~ group + band + group * band
            aov = pg.anova(data=long, dv='r', between=['group', 'band'])
            print(aov[['Source', 'F', 'p_unc', 'np2']].to_string(index=False))
        except Exception as e:
            print(f'  ERROR: {e}')

Two-way ANOVA: group * band for each feature decoding and both pipelines

  envelope — not standardized
      Source          F         p_unc      np2
       group  71.476222  4.323391e-17 0.023347
        band 323.749563 1.042620e-231 0.302217
group * band   4.777949  7.658158e-04 0.006351
    Residual        NaN           NaN      NaN

  onsets — not standardized
      Source          F         p_unc      np2
       group  59.860241  1.382253e-14 0.019627
        band 481.279646 1.130916e-320 0.391673
group * band  10.734475  1.224998e-08 0.014157
    Residual        NaN           NaN      NaN

  pitch — not standardized
      Source         F        p_unc      np2
       group  2.820849 9.315166e-02 0.000943
        band 68.979111 6.247218e-56 0.084484
group * band  0.685270 6.021044e-01 0.000916
    Residual       NaN          NaN      NaN

  centroid — not standardized
      Source          F         p_unc      np2
       group  22.560834  2.133862e-06 0.007641
        band 183.56

In [4]:
# Two-way ANOVA with group (musicians vs non-musicians) and band (delta, theta, alpha, beta, wideband) as factors (for each feature encoding for both pipelines)
print('Two-way ANOVA: group * band for each feature encoding and both pipelines')

# Both pipelines
for pipeline in ['not standardized', 'standardized']:
    
    # Use the same feature order as before
    feats = feat_orders[pipeline]
    
    # Get encoding rows
    enc_df = df[(df['direction'] == 'encoding') & (df['pipeline'] == pipeline)]
    
    # One ANOVA per single feature
    for feat in feats:
        feat_df = enc_df[enc_df['combo_name'] == feat]
        
        # Reshape to long format (one row per trial)
        long = to_long(feat_df) # use funciton from previous cell
        
        # Print the result for this (feature and pipeline)
        print(f'\n  {feat} — {pipeline}')
        try:
            # Two-way ANOVA: r ~ group + band + group * band
            aov = pg.anova(data=long, dv='r', between=['group', 'band'])
            print(aov[['Source', 'F', 'p_unc', 'np2']].to_string(index=False))
        except Exception as e:
            print(f'  ERROR: {e}')

Two-way ANOVA: group * band for each feature encoding and both pipelines

  envelope — not standardized
      Source          F        p_unc      np2
       group  11.499923 7.050276e-04 0.003831
        band 117.974279 1.462386e-93 0.136312
group * band   1.413097 2.269444e-01 0.001887
    Residual        NaN          NaN      NaN

  onsets — not standardized
      Source          F        p_unc      np2
       group   8.690031 3.224321e-03 0.002898
        band 113.538433 3.069064e-90 0.131862
group * band   2.140678 7.328697e-02 0.002856
    Residual        NaN          NaN      NaN

  pitch — not standardized
      Source         F        p_unc      np2
       group  0.005060 9.432952e-01 0.000002
        band 13.581503 5.652860e-11 0.018023
group * band  0.199043 9.389392e-01 0.000269
    Residual       NaN          NaN      NaN

  centroid — not standardized
      Source          F        p_unc      np2
       group   5.710887 1.692161e-02 0.001906
        band 114.813570 3.39140

In [5]:
# One-way ANOVA for answering does adding features (1-5) improve encoding results (for both pipelines and all bands)

print('One way ANOVA: n_features (1-5) for encoding')

# Both pipelines
for pipeline in ['not standardized', 'standardized']:
    
    # Get encoding rows
    enc_df = df[(df['direction'] == 'encoding') & (df['pipeline'] == pipeline)]
    
    # One ANOVA per band
    for band in bands:
        band_df = enc_df[enc_df['band'] == band]
        
        # Build long format with n_features as the factor (following the pattern of the function in cell 1)
        rows = []
        for _, row in band_df.iterrows():
            
            # For all trials
            for t in range(1, 31):
                val = row[f'r_trial_{t}']
                
                # Add all needed info for ANOVA (trial r's, subject, number of features in a combo)
                rows.append({
                    'subject': row['subject'],
                    'n_features': str(int(row['n_features'])),
                    'r': val,
                })
        long = pd.DataFrame(rows)
        
        # Print the result for this (band, pipeline)
        print(f'\n  {band} — {pipeline}')
        try:
            # One-way ANOVA: r ~ n_features
            aov = pg.anova(data=long, dv='r', between='n_features')
            print(aov[['Source', 'F', 'p_unc', 'np2']].to_string(index=False))
            
            # Post-hoc pairwise tests (Bonferroni corrected) to see which pairs differ
            ph = pg.pairwise_tests(data=long, dv='r', between='n_features', padjust='bonf')
            
            # Find the corrected p-value column (pingouin names it slightly differently across different versions)
            p_col = [c for c in ph.columns if 'p-corr' in c.lower() or 'p_corr' in c.lower()]
            if p_col:
                
                # Keep only significant pairs
                sig = ph[ph[p_col[0]] < 0.05]
                if not sig.empty:
                    print('  Significant pairs (Bonferroni):')
                    for _, row in sig.iterrows():
                        print(f'    {row["A"]} vs {row["B"]}: p={row[p_col[0]]:.6f}') # 10 rows in total but only show significant ones
        except Exception as e:
            print(f'  ERROR: {e}')

One way ANOVA: n_features (1-5) for encoding

  delta — not standardized
    Source        F        p_unc      np2
n_features 17.36667 3.126854e-14 0.003722
  Significant pairs (Bonferroni):
    1 vs 4: p=0.000000
    1 vs 5: p=0.000023
    2 vs 3: p=0.001230
    2 vs 4: p=0.000000
    2 vs 5: p=0.000003
    3 vs 4: p=0.001697
    3 vs 5: p=0.004046

  theta — not standardized
    Source         F        p_unc      np2
n_features 116.76634 1.550010e-98 0.024502
  Significant pairs (Bonferroni):
    1 vs 2: p=0.043249
    1 vs 3: p=0.000006
    1 vs 4: p=0.000000
    1 vs 5: p=0.000000
    2 vs 3: p=0.000000
    2 vs 4: p=0.000000
    2 vs 5: p=0.000000
    3 vs 4: p=0.000000
    3 vs 5: p=0.000000
    4 vs 5: p=0.000000

  alpha — not standardized
    Source         F        p_unc      np2
n_features 88.262542 1.981985e-74 0.018662
  Significant pairs (Bonferroni):
    1 vs 2: p=0.020569
    1 vs 3: p=0.002856
    1 vs 4: p=0.000000
    1 vs 5: p=0.000000
    2 vs 3: p=0.000000
    2 v

In [6]:
# Two-way ANOVA for answering does processing features (raw vs processed (without mel and mfcc3)) improve results (with pipeline and band as factors for both directions)
# Use all feature combinations for encoding (that do not include mel and mfcc3)

print('Two-way ANOVA: pipeline * band for encoding and decoding')

# Both directions
for direction in ['decoding', 'encoding']:
    print(f'\n {direction}')
    
    # Get rows for this direction
    dir_df = df[df['direction'] == direction]

    # Use only feature combos that exist in both pipelines (drop mel and mfcc3 combos since they are pipeline-specific)
    dir_df = dir_df[~dir_df['combo_name'].str.contains('mel|mfcc3')]

    # Build long format with pipeline and band as factors
    rows = []
    for _, row in dir_df.iterrows(): # following the pattern of the function in cell 1
        
        # For all trials
        for t in range(1, 31):
            val = row[f'r_trial_{t}']
                
            # Add all needed info for ANOVA (trial r's, pipeline, band)
            rows.append({
                'pipeline': row['pipeline'],
                'band': row['band'],
                'r': val,
            })
    long = pd.DataFrame(rows)
    
    try:
        # Two-way ANOVA: r ~ pipeline + band + pipeline * band
        aov = pg.anova(data=long, dv='r', between=['pipeline', 'band'])
        print(aov[['Source', 'F', 'p_unc', 'np2']].to_string(index=False))
    except Exception as e:
        print(f'  ERROR: {e}')

Two-way ANOVA: pipeline * band for encoding and decoding

 decoding
         Source           F         p_unc      np2
       pipeline  671.792128 4.275141e-146 0.027373
           band 1121.118195  0.000000e+00 0.158158
pipeline * band   90.250288  2.810967e-76 0.014898
       Residual         NaN           NaN      NaN

 encoding
         Source           F        p_unc      np2
       pipeline  219.305802 1.466515e-49 0.002432
           band 3211.375095 0.000000e+00 0.124950
pipeline * band   13.931336 2.307906e-11 0.000619
       Residual         NaN          NaN      NaN


In [7]:
# Two-way ANOVA for answering does processing features (raw vs processed (without mel and mfcc3)) improve results (with pipeline and band as factors for both directions)
# Use only single features for encoding (that do not include mel and mfcc3)
print('Two-way ANOVA: pipeline * band for encoding and decoding')

# Both directions
for direction in ['decoding', 'encoding']:
    print(f'\n {direction}')
    
    # Get rows for this direction
    dir_df = df[df['direction'] == direction]
    
    # Keep only the 4 single features shared between pipelines (drop mel and mfcc3 since they are pipeline-specific)
    dir_df = dir_df[dir_df['combo_name'].isin(['envelope', 'onsets', 'pitch', 'centroid'])]
    
    # Build long format with pipeline and band as factors
    rows = []
    for _, row in dir_df.iterrows(): # following the pattern of the function in cell 1
        
        # For all trials
        for t in range(1, 31):
            val = row[f'r_trial_{t}']
                
            # Add all needed info for ANOVA (trial r's, pipeline, band)
            rows.append({
                'pipeline': row['pipeline'],
                'band': row['band'],
                'r': val,
            })
    long = pd.DataFrame(rows)
    
    try:
        # Two-way ANOVA: r ~ pipeline + band + pipeline * band
        aov = pg.anova(data=long, dv='r', between=['pipeline', 'band'])
        print(aov[['Source', 'F', 'p_unc', 'np2']].to_string(index=False))
    except Exception as e:
        print(f'  ERROR: {e}')

Two-way ANOVA: pipeline * band for encoding and decoding

 decoding
         Source           F         p_unc      np2
       pipeline  671.792128 4.275141e-146 0.027373
           band 1121.118195  0.000000e+00 0.158158
pipeline * band   90.250288  2.810967e-76 0.014898
       Residual         NaN           NaN      NaN

 encoding
         Source          F        p_unc      np2
       pipeline 262.330356 1.092716e-58 0.010830
           band 678.160457 0.000000e+00 0.101701
pipeline * band   8.655656 5.614953e-07 0.001443
       Residual        NaN          NaN      NaN


In [8]:
# Linear regression (with binary feature indicators) to see which features improve (or hurt) encoding r-values (for both pipelines and all bands)
print('Linear regression: feature contribution to encoding')

# Both pipelines
for pipeline in ['not standardized', 'standardized']:
    
    # Get encoding rows
    enc_df = df[(df['direction'] == 'encoding') & (df['pipeline'] == pipeline)]
    
    # All single features that appear in this pipeline's combos
    all_feats = feat_orders[pipeline] # defined in cell 0
    
    # Build long format with binary indicators (1 if feature is in the combo, 0 if not)
    rows = []
    for _, row in enc_df.iterrows():
        
        # One indicator per feature
        feat_flags = {f'has_{f}': 1 if f in row['combo_name'].split('+') else 0 for f in all_feats}
        
        # For all trials
        for t in range(1, 31):
            val = row[f'r_trial_{t}']
            
            # Add all needed info for the regression (trial r's, subject, group, band, feature flags)
            row_dict = {
                'subject': row['subject'],
                'group': row['group'],
                'band': row['band'],
                'r': val,
            }
            row_dict.update(feat_flags) # Merge
            rows.append(row_dict)
    long_enc = pd.DataFrame(rows)
    
    # One regression per band
    for band in bands:
        band_long = long_enc[long_enc['band'] == band]
        
        # Regress r on the feature flags
        formula = 'r ~ ' + ' + '.join([f'has_{f}' for f in all_feats])
        
        # Print the result for this band and pipeline
        print(f'\n  {band} — {pipeline}')
        try:
            model = smf.ols(formula, band_long)
            result = model.fit()
            
            # Print just coefficients and significance per feature
            print(f'  {"Feature":<15} {"Coef":>10} {"p":>10} {"Sig":>5}')
            print(f'  {"-" * 44}')
            for feat in all_feats:
                coef = result.params[f'has_{feat}']
                p = result.pvalues[f'has_{feat}']
                
                # Add significance stars
                sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                print(f'  {feat:<15} {coef:>10.4f} {p:>10.4f} {sig:>5}')
        except Exception as e:
            print(f'  ERROR: {e}')

Linear regression: feature contribution to encoding

  delta — not standardized
  Feature               Coef          p   Sig
  --------------------------------------------
  envelope            0.0002     0.4139    ns
  onsets              0.0013     0.0000   ***
  pitch              -0.0003     0.2875    ns
  centroid           -0.0001     0.6069    ns
  mel                -0.0058     0.0000   ***

  theta — not standardized
  Feature               Coef          p   Sig
  --------------------------------------------
  envelope            0.0033     0.0000   ***
  onsets              0.0030     0.0000   ***
  pitch              -0.0015     0.0000   ***
  centroid            0.0015     0.0000   ***
  mel                -0.0221     0.0000   ***

  alpha — not standardized
  Feature               Coef          p   Sig
  --------------------------------------------
  envelope            0.0022     0.0000   ***
  onsets              0.0024     0.0000   ***
  pitch              -0.0012     